# Forecasting
Este notebook carga datos de inferencia para posteriores análisis y predicciones.

In [3]:
# Imports principales (basados en entrenamiento.ipynb)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays
import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar

# Cargar ventas históricas, datos de inferencia y competencia
ventas_hist = pd.read_csv('../data/raw/entrenamiento/ventas.csv')
ventas_hist['fecha'] = pd.to_datetime(ventas_hist['fecha'])
ventas_hist['is_inferencia'] = False
vent_inf = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
vent_inf['fecha'] = pd.to_datetime(vent_inf['fecha'])
vent_inf['is_inferencia'] = True
competencia_df = pd.read_csv('../data/raw/entrenamiento/competencia.csv')
competencia_df['fecha'] = pd.to_datetime(competencia_df['fecha'])

# Concatenar histórico + inferencia para poder calcular lags con histórico
df_all = pd.concat([ventas_hist, vent_inf], ignore_index=True, sort=False)

# Merge competencia (left) para añadir precios competidores cuando existan
df_all = pd.merge(df_all, competencia_df, on=['fecha','producto_id'], how='left')

# Variables temporales y eventos (mismo proceso que en entrenamiento.ipynb)
spain_holidays = holidays.Spain()
df_all['año'] = df_all['fecha'].dt.year
df_all['mes'] = df_all['fecha'].dt.month
df_all['mes_nombre'] = df_all['fecha'].dt.month_name()
df_all['dia_mes'] = df_all['fecha'].dt.day
df_all['dia_semana'] = df_all['fecha'].dt.day_name()
df_all['dia_semana_num'] = df_all['fecha'].dt.weekday
df_all['es_fin_de_semana'] = df_all['dia_semana_num'] >= 5
df_all['es_festivo'] = df_all['fecha'].isin(spain_holidays)
df_all['nombre_festivo'] = df_all['fecha'].map(spain_holidays)
df_all['es_dia_lectivo'] = ~df_all['es_festivo'] & ~df_all['es_fin_de_semana']
df_all['trimestre'] = df_all['fecha'].dt.quarter
df_all['semana_año'] = df_all['fecha'].dt.isocalendar().week
df_all['es_inicio_mes'] = df_all['dia_mes'] <= 7
df_all['es_primer_dia_mes'] = df_all['dia_mes'] == 1
df_all['es_fin_mes'] = df_all['dia_mes'] >= 25
df_all['es_ultimo_dia_mes'] = df_all['fecha'] == (df_all['fecha'] + pd.offsets.MonthEnd(0))
def get_last_friday(year):
    nov = pd.date_range(start=f'{year}-11-01', end=f'{year}-11-30', freq='D')
    fridays = nov[nov.weekday == 4]
    return fridays[-1] if len(fridays) else None
black_fridays = [get_last_friday(y) for y in df_all['año'].unique()]
black_fridays = [d for d in black_fridays if d is not None]
cyber_mondays = [d + pd.Timedelta(days=3) for d in black_fridays]
df_all['black_friday'] = df_all['fecha'].isin(black_fridays)
df_all['cyber_monday'] = df_all['fecha'].isin(cyber_mondays)
df_all['es_ultimo_tramo_mes'] = df_all['dia_mes'] >= 24

# Crear lags y rolling usando todo el histórico (para que las filas de inferencia tengan lags)
df_all = df_all.sort_values(['producto_id', 'fecha']).reset_index(drop=True)
for lag in range(1,8):
    df_all[f'unidades_vendidas_lag_{lag}'] = df_all.groupby(['producto_id'])['unidades_vendidas'].shift(lag)
df_all['unidades_vendidas_roll_7'] = df_all.groupby(['producto_id'])['unidades_vendidas'].transform(lambda x: x.rolling(window=7, min_periods=7).mean())
nuevas = [f'unidades_vendidas_lag_{lag}' for lag in range(1,8)] + ['unidades_vendidas_roll_7']

# Descuento y precios de competencia
df_all['descuento_porcentaje'] = ((df_all['precio_venta'] - df_all['precio_base']) / df_all['precio_base']) * 100
if set(['Amazon','Decathlon','Deporvillage']).issubset(df_all.columns):
    df_all['precio_competencia'] = df_all[['Amazon','Decathlon','Deporvillage']].mean(axis=1)
else:
    df_all['precio_competencia'] = np.nan
df_all['ratio_precio'] = df_all['precio_venta'] / df_all['precio_competencia']
for c in ['Amazon','Decathlon','Deporvillage']:
    if c in df_all.columns:
        df_all = df_all.drop(columns=[c])

# One-hot encoding para nombre/categoria/subcategoria (copias *_h)
df_all['nombre_h'] = df_all['nombre']
df_all['categoria_h'] = df_all['categoria']
df_all['subcategoria_h'] = df_all['subcategoria']
dummies = pd.get_dummies(df_all[['nombre_h','categoria_h','subcategoria_h']].astype(str), prefix=['nombre_h','categoria_h','subcategoria_h'])
df_all = pd.concat([df_all, dummies], axis=1)

# Seleccionar sólo las filas de inferencia y quedarnos sólo con noviembre
inferencia_df = df_all[df_all['is_inferencia'] == True].copy()
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].reset_index(drop=True)

# Mantener todas las filas de inferencia (no eliminar por NaNs en lags),
# porque queremos conservar el contenido original y las transformaciones aplicadas.
# Si se desea filtrar para el modelo, hacerlo más adelante antes de predecir.
inferencia_df = inferencia_df.reset_index(drop=True)

# Guardar inferencia transformada
from pathlib import Path
output_dir = Path('..') / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'inferencia_df_transformado.csv'
inferencia_df.to_csv(output_path, index=False, encoding='utf-8')
print(f'Guardado inferencia transformada en: {output_path} - filas: {len(inferencia_df)}')
display(inferencia_df.head())

Guardado inferencia transformada en: ..\data\processed\inferencia_df_transformado.csv - filas: 720


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
1,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
2,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
3,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False
4,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,...,False,False,False,False,False,False,False,False,True,False


In [4]:
inferencia_df.shape

(720, 94)

In [5]:
inferencia_df.fecha.unique()

<DatetimeArray>
['2025-11-01 00:00:00', '2025-11-02 00:00:00', '2025-11-03 00:00:00',
 '2025-11-04 00:00:00', '2025-11-05 00:00:00', '2025-11-06 00:00:00',
 '2025-11-07 00:00:00', '2025-11-08 00:00:00', '2025-11-09 00:00:00',
 '2025-11-10 00:00:00', '2025-11-11 00:00:00', '2025-11-12 00:00:00',
 '2025-11-13 00:00:00', '2025-11-14 00:00:00', '2025-11-15 00:00:00',
 '2025-11-16 00:00:00', '2025-11-17 00:00:00', '2025-11-18 00:00:00',
 '2025-11-19 00:00:00', '2025-11-20 00:00:00', '2025-11-21 00:00:00',
 '2025-11-22 00:00:00', '2025-11-23 00:00:00', '2025-11-24 00:00:00',
 '2025-11-25 00:00:00', '2025-11-26 00:00:00', '2025-11-27 00:00:00',
 '2025-11-28 00:00:00', '2025-11-29 00:00:00', '2025-11-30 00:00:00']
Length: 30, dtype: datetime64[us]